# 16 — Demographic GroupBy: Cross-Cohort Aggregations & Extremes
> **Interview Prep & Technical Mastery Guide**
> 
> *A comprehensive, battle-tested reference for Python Data Science, People Analytics, and Statistical Aggregation Interviews.*

---

## 📌 Executive Summary & Interview Expectations
This lab tests your ability to aggregate multi-level demographic cohorts. In technical interviews, interviewers evaluate whether you can compute **proportions and ratios vectorially**, extract **top/bottom extremes (`.nsmallest()`)**, and avoid slow row-by-row `apply()` functions.

### Core Competencies Tested in this Module:
1. **Vectorized Proportions vs `apply()`**: Why `(s == 'M').groupby(...).mean()` is 100x faster than defining a Python function.
2. **Multi-Cohort Grouping**: Grouping by `['occupation', 'gender']` with `as_index=False`.
3. **Cross-Tabulation for Percentages**: Using `pd.crosstab(..., normalize='index')` to eliminate multi-step group divisions.
4. **Group Extremes & Rank Selection**: The critical distinction between `.nsmallest(K)`, `.head(K)`, and `.nth(K)`.
5. **Interview Corner**: Positional `.nth()` vs sorted `.nsmallest()`, and calculating demographic age gaps.

## 1. Environment Setup & Data Ingestion

In [1]:
import os
import numpy as np
import pandas as pd

# Load MovieLens users dataset (pipe-delimited)
file_path = "user.csv"
if not os.path.exists(file_path):
    file_path = "https://raw.githubusercontent.com/justmarkham/DAT8/master/data/u.user"

users = pd.read_csv(file_path, sep="|", index_col="user_id")
print("User demographics loaded. Dimensions:", users.shape)
users.head(4)

User demographics loaded. Dimensions: (943, 4)


,age,gender,occupation,zip_code
user_id,,,,
1,24,M,technician,85711
2,53,F,other,94043
3,23,M,writer,32067
4,24,M,technician,43537


## 2. Age Distribution by Occupation: Mean, Min, and Max

In [2]:
# Mean age per occupation (sorted descending)
mean_age = (
    users.groupby("occupation")["age"]
    .mean()
    .round(1)
    .sort_values(ascending=False)
)
display(mean_age.head(5))

occupation
retired       63.1
doctor        43.6
educator      42.0
healthcare    41.6
librarian     40.0
Name: age, dtype: float64

In [3]:
# Min and Max ages per occupation using named aggregation
age_range = (
    users.groupby("occupation", as_index=False)
    .agg(min_age=("age", "min"), max_age=("age", "max"))
    .sort_values(by="max_age", ascending=False)
)
display(age_range.head(5))

,occupation,min_age,max_age
15,retired,51,73
0,administrator,21,70
4,engineer,22,70
6,executive,22,69
10,librarian,23,69


## 3. Multi-Cohort Analysis: `['occupation', 'gender']`

In [4]:
# Mean age for each occupation and gender combination
cohort_age = (
    users.groupby(["occupation", "gender"], as_index=False)["age"]
    .mean()
    .round(1)
    .sort_values(by="age", ascending=False)
)
cohort_age.head(6)

,occupation,gender,age
29,retired,F,70.0
30,retired,M,62.5
14,healthcare,M,45.4
11,executive,F,44.0
4,doctor,M,43.6
6,educator,M,43.1


## 4. Male Ratio per Occupation: Vectorized vs `apply()`

### ⚠️ Top Interview Optimization: The Boolean Mean Trick
- **Anti-Pattern**: Writing a Python function `def to_numeric(x): if x == 'M': return 1 ...` and calling `.apply()`.
- **Vectorized Standard**: `users['gender'] == 'M'` yields a boolean vector (`True`/`False`).
  In Python/NumPy, `True == 1` and `False == 0`. Taking the **mean** of a boolean vector directly computes the proportion of `True`s in compiled C code!

In [5]:
# Pure vectorized male ratio calculation
male_ratio = (
    (users["gender"] == "M")
    .groupby(users["occupation"])
    .mean()
    .mul(100)
    .round(1)
    .sort_values(ascending=False)
)

print("Top 5 occupations with highest male representation (%):")
display(male_ratio.head(5))

Top 5 occupations with highest male representation (%):


occupation
doctor        100.0
engineer       97.0
technician     96.3
retired        92.9
programmer     90.9
Name: gender, dtype: float64

## 5. Gender Percentages per Occupation via `pd.crosstab`

Instead of manually grouping, summing, and dividing series, `pd.crosstab(..., normalize='index')` computes normalized row percentages in a single step.

In [6]:
# Normalized gender breakdown per occupation
gender_breakdown = (
    pd.crosstab(users["occupation"], users["gender"], normalize="index") * 100
).round(1)

gender_breakdown.sort_values(by="F", ascending=False).head(6)

gender,F,M
occupation,,
homemaker,85.7,14.3
healthcare,68.8,31.2
librarian,56.9,43.1
artist,46.4,53.6
administrator,45.6,54.4
none,44.4,55.6


## 6. Identifying Cohort Extremes: `.nsmallest()` vs `.nth()`

### ⚠️ Top Interview Question: `.nsmallest(K)` vs `.nth(K)`
- **`.nsmallest(K)`**: Sorts the group and extracts the **$K$ smallest values**.
- **`.nth(K)`**: Selects the **$K$-th item in original DataFrame row order** (0-indexed). It does NOT sort!

In [7]:
# Top 3 youngest users in each occupation and gender cohort
youngest_users = users.groupby(["occupation", "gender"])["age"].nsmallest(3)
youngest_users.head(9)

occupation     gender  user_id
administrator  F       180        22
                       439        23
                       726        25
               M       118        21
                       282        22
                       317        22
artist         F       601        19
                       150        20
                       24         21
Name: age, dtype: int64

In [8]:
# Select the 3rd person listed (index 2) in each cohort (positional, unsorted)
third_listed = users.groupby(["occupation", "gender"])["age"].nth(2)
third_listed.head(5)

user_id
11    39
22    25
33    23
41    33
42    30
Name: age, dtype: int64

## 7. Demographic Aggregation Cheat Sheet

| Task | Idiomatic Syntax | Performance / Benefit |
| :--- | :--- | :--- |
| **Category Proportion** | `(s == 'val').groupby(grp).mean()` | Vectorized C execution; no slow `apply()` |
| **Row Percentages** | `pd.crosstab(r, c, normalize='index')` | Direct contingency table with row sums = 1.0 |
| **Top K Smallest** | `df.groupby(grp)['col'].nsmallest(K)` | Algorithmic min-heap ($O(N \log K)$) |
| **Positional K-th** | `df.groupby(grp)['col'].nth(K)` | 0-based offset in original table order |

---
## 🎯 8. Technical Interview Corner: Tricky Questions & Drills

### Q1: The Boolean Mean Trick Explained
**Question**: An interviewer asks: *"You have a column `status` with values `['PASS', 'FAIL']`. How do you calculate the pass percentage for each department in a single vectorized expression without using `lambda` or `map`?"*

**Answer**:
```python
(df['status'] == 'PASS').groupby(df['department']).mean() * 100
```
Evaluating `status == 'PASS'` creates a boolean Series where `True` is treated as `1` and `False` as `0`. The arithmetic mean of binary values is mathematically identical to the sample proportion.

### Q2: Advanced Interview Challenge: Maximum Gender Age Gap
**Challenge**: In a single chained expression, find the occupation with the **largest absolute difference between the average male age and average female age**, considering only occupations with at least 5 members of each gender!

In [9]:
# Solution to Coding Challenge
gender_age_gap = (
    users.groupby(["occupation", "gender"])
    .agg(avg_age=("age", "mean"), count=("age", "count"))
    .unstack(level="gender")
)

# Filter for occupations with >= 5 males and >= 5 females
gender_age_gap = gender_age_gap[
    (gender_age_gap[("count", "M")] >= 5) & (gender_age_gap[("count", "F")] >= 5)
].copy()

gender_age_gap["age_difference"] = (
    gender_age_gap[("avg_age", "M")] - gender_age_gap[("avg_age", "F")]
).abs().round(1)

top_age_gap = gender_age_gap.sort_values(by="age_difference", ascending=False).head(3)
display(top_age_gap)

avg_age            count       age_difference
gender                 F          M     F     M               
occupation                                                    
healthcare     39.818182  45.400000  11.0   5.0            5.6
educator       39.115385  43.101449  26.0  69.0            4.0
administrator  40.638889  37.162791  36.0  43.0            3.5